In [9]:
import numpy as np
import time
from coppeliasim_zmqremoteapi_client import RemoteAPIClient

client = RemoteAPIClient()
sim = client.getObject('sim')
client.setStepping(False)
sim.startSimulation()
print("Connected to CoppeliaSim and simulation started!")


Connected to CoppeliaSim and simulation started!


In [10]:
def FK(theta, alpha, a, d):
    """
    Compute the homogeneous transformation matrix using Denavit-Hartenberg parameters.
    
    Parameters:
    -----------
    theta : float
        Joint angle (radians)
    alpha : float
        Link twist angle (radians)
    a : float
        Link length
    d : float
        Joint offset
    
    Returns:
    --------
    T : np.ndarray
        4x4 homogeneous transformation matrix
    """
    # Precompute trigonometric values
    Cth = np.cos(theta)
    Sth = np.sin(theta)
    Calp = np.cos(alpha)
    Salp = np.sin(alpha)

    T = np.array([
        [Cth, -Sth * Calp,  Sth * Salp,  a * Cth],
        [Sth,  Cth * Calp, -Cth * Salp,  a * Sth],
        [0,    Salp,        Calp,        d],
        [0,    0,           0,           1]
    ])

    return T


In [11]:
# KUKA LWR robot dimensions (from specification PDF)
L = 0.40   # d3 - link length (meters)
M = 0.39   # d5 - link length (meters)

# Initial joint configuration (radians)
# 7 DOF configuration for KUKA LWR
q = [0, np.pi/2, np.pi/2, -np.pi/2, 0, np.pi/2, 0]

print("Robot Parameters:")
print(f"Link length L (d3): {L} m")
print(f"Link length M (d5): {M} m")
print(f"Joint angles (q): {np.round(q, 4)}")


Robot Parameters:
Link length L (d3): 0.4 m
Link length M (d5): 0.39 m
Joint angles (q): [ 0.      1.5708  1.5708 -1.5708  0.      1.5708  0.    ]


In [12]:
# Compute individual transformation matrices for each joint
T0_1 = FK(q[0],  np.pi/2, 0, 0)
T1_2 = FK(q[1], -np.pi/2, 0, 0)
T2_3 = FK(q[2], -np.pi/2, 0, L)
T3_4 = FK(q[3],  np.pi/2, 0, 0)
T4_5 = FK(q[4],  np.pi/2, 0, M)
T5_6 = FK(q[5], -np.pi/2, 0, 0)
T6_7 = FK(q[6],  0,       0, 0)

# Chain all transformations to compute end-effector pose
T0_7 = T0_1 @ T1_2 @ T2_3 @ T3_4 @ T4_5 @ T5_6 @ T6_7

print("Analytical Transformation Matrix (T0_7):")
print(np.round(T0_7, 4))
print("\nEnd-effector position (analytical):", np.round(T0_7[:3, 3], 4))


Analytical Transformation Matrix (T0_7):
[[-0.   -0.    1.   -0.4 ]
 [-1.    0.   -0.   -0.39]
 [-0.   -1.   -0.    0.  ]
 [ 0.    0.    0.    1.  ]]

End-effector position (analytical): [-0.4  -0.39  0.  ]


In [13]:
# Get object handles from CoppeliaSim
joint_names = [f'joint_{i+1}' for i in range(7)]
joint_handles = [sim.getObjectHandle(name) for name in joint_names]

# Get end-effector dummy object
EE = sim.getObjectHandle("EE_Dummy")

print("Joint Handles (CoppeliaSim):", joint_handles)
print("End-Effector Handle:", EE)
print("\nSimulation is running...")


Joint Handles (CoppeliaSim): [17, 20, 23, 26, 29, 32, 35]
End-Effector Handle: 38

Simulation is running...


In [14]:
# Send target joint positions to CoppeliaSim
for i in range(7):
    sim.setJointTargetPosition(joint_handles[i], q[i])

print("Joint commands sent to simulation")

# Wait for simulation to settle
time.sleep(1)

# Retrieve the end-effector transformation matrix from CoppeliaSim
T_EE_sim = sim.getObjectMatrix(EE, -1)

# Convert to standard 4x4 homogeneous transformation format
T_EE_sim = np.array(T_EE_sim).reshape(3, 4)
T_EE_sim = np.vstack((T_EE_sim, [0, 0, 0, 1]))

print("Simulated Transformation Matrix (T_EE):")
print(np.round(T_EE_sim, 4))
print("\nEnd-effector position (simulated):", np.round(T_EE_sim[:3, 3], 4))


Joint commands sent to simulation
Simulated Transformation Matrix (T_EE):
[[-0.0011  0.0038 -1.     -0.7552]
 [ 1.     -0.0089 -0.0012  0.2646]
 [-0.0089 -1.     -0.0038  0.3053]
 [ 0.      0.      0.      1.    ]]

End-effector position (simulated): [-0.7552  0.2646  0.3053]


In [15]:
# Calculate errors between analytical and simulated transformations
position_error = np.linalg.norm(T0_7[:3, 3] - T_EE_sim[:3, 3])
rotation_error = np.linalg.norm(T0_7[:3, :3] - T_EE_sim[:3, :3])

print("=" * 50)
print("ERROR ANALYSIS")
print("=" * 50)
print(f"\nPosition Error (Euclidean norm): {position_error:.6f} m")
print(f"Rotation Error (Frobenius norm): {rotation_error:.6f}")

print("\n" + "=" * 50)
print("TRANSFORMATION MATRICES COMPARISON")
print("=" * 50)
print("\nAnalytical (T0_7):")
print(np.round(T0_7, 4))
print("\nSimulated (T_EE):")
print(np.round(T_EE_sim, 4))

# Validate the match
if position_error < 1e-4 and rotation_error < 1e-4:
    print("\n✓ Analytical and simulated results match within tolerance!")
else:
    print("\n⚠ Discrepancy between analytical and simulated results")


ERROR ANALYSIS

Position Error (Euclidean norm): 0.804923 m
Rotation Error (Frobenius norm): 2.828427

TRANSFORMATION MATRICES COMPARISON

Analytical (T0_7):
[[-0.   -0.    1.   -0.4 ]
 [-1.    0.   -0.   -0.39]
 [-0.   -1.   -0.    0.  ]
 [ 0.    0.    0.    1.  ]]

Simulated (T_EE):
[[-0.0011  0.0038 -1.     -0.7552]
 [ 1.     -0.0089 -0.0012  0.2646]
 [-0.0089 -1.     -0.0038  0.3053]
 [ 0.      0.      0.      1.    ]]

⚠ Discrepancy between analytical and simulated results
